# 🛰️ Update alternative data — insider, analyst ratings, short interest

A peer of `01_universe_screener` and `02_backfill`, but for the **non-OHLCV**
signals that drive the `informed_flow` strategy. Everything here is **this
repo's own ingestion** — SEC EDGAR (Form 4 insider), Yahoo Finance (analyst
rating changes), FINRA (short interest) — writing to the **shared TimescaleDB**.
No dependency on the sibling `stock-trading` repo.

Point-in-time by construction: every row is stamped `known_at` (SEC filing
acceptance, FINRA dissemination, rating announcement), so features never see a
value before it was public. Writes UPSERT on natural keys, so re-running is
safe and cheap.

Flow: **set parameters → update each dataset → check freshness → check schema.**

## 1 · Parameters

In [ ]:
UNIVERSE = "LiquidLeaders"     # names to keep fresh (the informed_flow universe)
DATASETS = ["insider", "ratings", "short_interest"]
SINCE    = "2023-01-01"          # history depth (the strategy trains from 2020)
SYMBOLS  = None                  # None = the whole universe; or a short list to smoke-test

# SEC EDGAR REQUIRES a contact email in the User-Agent or it returns 403.
# Put your own name + email here (SEC uses it only to reach you if you cause load).
import os
os.environ.setdefault("TRADING_SEC_USER_AGENT", "your-name your-email@example.com")

## 2 · Set up

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel, or run
#   uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from datetime import date
import pandas as pd
from sqlalchemy import text

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import jobs

settings = load_settings()
engine = make_engine(settings)
symbols = SYMBOLS or settings.universe_symbols(UNIVERSE)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")
print(f"{UNIVERSE}: {len(symbols)} symbols; datasets: {DATASETS}")

## 3 · Update each dataset

Each dataset runs as **one paced `tsim altdata` subprocess** (mirroring the
backfill handoff). SEC and FINRA rate-limit per IP, so — like the IBKR rule —
there is deliberately **no fan-out**; per-symbol progress streams below. Insider
is the slow one (a request per Form 4 filing); ratings and short interest are
quick. Re-running only writes rows not already stored (idempotent UPSERT), so
this is safe to run daily.

In [ ]:
for ds in DATASETS:
    print(f"\n=== {ds} ===")
    jobs.run_altdata(ds, since=SINCE, universe=UNIVERSE, symbols=SYMBOLS)

## 4 · Freshness & coverage

Is the data fresh, and does it cover the universe? For each table: row count,
the most recent `known_at`, how many universe names are present, and how stale
the newest row is. Insider/ratings should be current to within days; short
interest is bi-monthly (FINRA), so a couple of weeks of staleness is normal.

In [ ]:
TABLES = {"insider": "insider_trades", "ratings": "analyst_ratings",
          "short_interest": "short_interest"}
uni = settings.universe_symbols(UNIVERSE)
rows = []
with engine.connect() as c:
    for label, tbl in TABLES.items():
        n = c.execute(text(f"SELECT count(*) FROM {tbl}")).scalar()
        last = c.execute(text(f"SELECT max(known_at) FROM {tbl}")).scalar()
        covered = c.execute(
            text(f"SELECT count(DISTINCT symbol) FROM {tbl} WHERE symbol = ANY(:s)"),
            {"s": uni}).scalar()
        rows.append({"dataset": label, "table": tbl, "rows": n,
                     "last_known_at": last, "coverage": f"{covered}/{len(uni)}"})
report = pd.DataFrame(rows)
today = pd.Timestamp(date.today(), tz="UTC")
report["stale_days"] = report["last_known_at"].apply(
    lambda t: (today - pd.Timestamp(t)).days if t is not None else None)
display(report)

In [ ]:
import plotly.graph_objects as go
from trading_simple.plots import SERIES

fig = go.Figure(go.Bar(
    x=report["dataset"], y=report["stale_days"].fillna(0),
    marker_color=SERIES[0], text=report["rows"], textposition="outside"))
fig.update_layout(
    title="Alt-data staleness — days since the newest known_at (lower = fresher)",
    template="plotly_white", height=360, yaxis_title="days stale")
fig.show()

## 5 · Schema sanity

Confirm the shared database still has each alt-data table with the columns the
feature layer (`trading_simple.features`) reads. These tables are defined in
this repo's `schema.sql` (added via `CREATE TABLE IF NOT EXISTS`, column-for-
column identical to the sibling's), so this also catches any drift if the
sibling repo altered them.

In [ ]:
EXPECTED = {
    "insider_trades":  ["symbol", "txn_type", "shares", "price", "traded_at", "known_at"],
    "analyst_ratings": ["symbol", "firm", "action", "grade_to", "rated_at", "known_at"],
    "short_interest":  ["symbol", "settlement_date", "days_to_cover", "known_at"],
}
with engine.connect() as c:
    for tbl, cols in EXPECTED.items():
        have = {r[0] for r in c.execute(text(
            "SELECT column_name FROM information_schema.columns WHERE table_name = :t"),
            {"t": tbl})}
        if not have:
            print(f"{tbl:18s} MISSING TABLE — run  uv run tsim init-db")
            continue
        missing = [col for col in cols if col not in have]
        print(f"{tbl:18s} {'OK' if not missing else 'missing ' + str(missing)}")

## Next steps

The data is fresh — open **`strategies/informed_flow/informed_flow.ipynb`** to
backtest the composite strategy that ranks stocks on these signals. To automate
this refresh, the same work runs headless:

```bash
uv run tsim altdata --dataset all --universe LiquidLeaders --since 2020-01-01
```